# Adaptive Adversarial Evaluation — Results Analysis

This notebook loads `results.json` produced by `run_experiment.py` and generates all five required deliverables:

| # | Deliverable |
|---|---|
| A | Toxic Failure Rate vs Token Budget |
| B | Tokens Required per Toxic Failure |
| C | Sessions Required per Toxic Failure |
| D | Turn Depth Before Failure |
| E | Comparative Model Resilience Table |

**Run first:** `python run_experiment.py --multi-run --output results.json`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

from analysis.visualize import (
    load_results, build_summary_df, build_turn_depth_series,
    graph_a, graph_b, graph_c, graph_d, table_e,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
RESULTS_PATH = '../results.json'
CHARTS_DIR = Path('charts')
CHARTS_DIR.mkdir(exist_ok=True)

experiments = load_results(RESULTS_PATH)
df = build_summary_df(experiments)
turns_df = build_turn_depth_series(experiments)

print(f'Loaded {len(experiments)} experiment(s)')
df

## Graph A — Toxic Failure Rate vs Token Budget

In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

fig, ax = plt.subplots(figsize=(9, 5))
for model, grp in df.groupby('model'):
    grp_sorted = grp.sort_values('budget')
    ax.plot(grp_sorted['budget'], grp_sorted['toxic_failure_rate'] * 100, marker='o', linewidth=2, label=model)
ax.set_xlabel('Token Budget')
ax.set_ylabel('Toxic Failure Rate (%)')
ax.set_title('Graph A — Toxic Failure Rate vs Token Budget', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.set_ylim(0, 105)
ax.legend(title='Target Model')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(CHARTS_DIR / 'graph_a.png', dpi=150, bbox_inches='tight')
plt.show()

## Graph B — Tokens Required per Toxic Failure

In [ ]:
plot_df = df.dropna(subset=['tokens_per_failure'])
if plot_df.empty:
    print('No toxic failures yet — run more budget tiers or use the lenient mock variant.')
else:
    pivot = plot_df.pivot_table(index='budget', columns='model', values='tokens_per_failure', aggfunc='mean')
    x = np.arange(len(pivot.index))
    width = 0.8 / max(len(pivot.columns), 1)
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, col in enumerate(pivot.columns):
        offset = (i - len(pivot.columns) / 2 + 0.5) * width
        bars = ax.bar(x + offset, pivot[col] / 1000, width=width * 0.9, label=col)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3, f'{h:.1f}k', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{int(b):,}' for b in pivot.index])
    ax.set_xlabel('Token Budget')
    ax.set_ylabel('Tokens per Failure (×1,000)')
    ax.set_title('Graph B — Tokens Required per Toxic Failure', fontweight='bold')
    ax.legend(title='Target Model')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(CHARTS_DIR / 'graph_b.png', dpi=150, bbox_inches='tight')
    plt.show()

## Graph C — Sessions Required per Toxic Failure

In [ ]:
plot_df = df.dropna(subset=['sessions_per_failure'])
if plot_df.empty:
    print('No toxic failures yet.')
else:
    models = sorted(plot_df['model'].unique())
    data_by_model = [plot_df.loc[plot_df['model'] == m, 'sessions_per_failure'].values for m in models]
    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(data_by_model, labels=models, patch_artist=True)
    colors = plt.cm.Set2(np.linspace(0, 1, len(models)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xlabel('Target Model')
    ax.set_ylabel('Sessions per Toxic Failure')
    ax.set_title('Graph C — Sessions Required per Toxic Failure', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(CHARTS_DIR / 'graph_c.png', dpi=150, bbox_inches='tight')
    plt.show()

## Graph D — Turn Depth Before Failure

In [ ]:
if turns_df.empty:
    print('No toxic failure sessions yet.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model in sorted(turns_df['model'].unique()):
        vals = turns_df.loc[turns_df['model'] == model, 'turns_to_failure']
        ax.hist(vals, bins=range(1, 12), alpha=0.6, label=model, edgecolor='white')
    ax.set_xlabel('Turns Before Toxic Failure')
    ax.set_ylabel('Count')
    ax.set_title('Graph D — Turn Depth Before Toxic Failure', fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(title='Target Model')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(CHARTS_DIR / 'graph_d.png', dpi=150, bbox_inches='tight')
    plt.show()

## Table E — Comparative Model Resilience

In [ ]:
summary = (
    df.groupby('model')
    .agg(
        Budgets_Tested=('budget', lambda x: ', '.join(f'{int(v):,}' for v in sorted(x))),
        Avg_Failure_Rate=('toxic_failure_rate', lambda x: f'{x.mean():.1%}'),
        Avg_Tokens_per_Failure=('tokens_per_failure', lambda x: f'{x.dropna().mean():,.0f}' if len(x.dropna()) > 0 else '—'),
        Avg_Turns=('avg_turns_to_failure', lambda x: f'{x.dropna().mean():.1f}' if len(x.dropna()) > 0 else '—'),
        Avg_Sessions_per_Failure=('sessions_per_failure', lambda x: f'{x.dropna().mean():.1f}' if len(x.dropna()) > 0 else '—'),
    )
    .reset_index()
    .rename(columns={'model': 'Model', 'Budgets_Tested': 'Budgets Tested',
                     'Avg_Failure_Rate': 'Avg Failure Rate',
                     'Avg_Tokens_per_Failure': 'Avg Tokens/Failure',
                     'Avg_Turns': 'Avg Turns/Failure',
                     'Avg_Sessions_per_Failure': 'Avg Sessions/Failure'})
)
summary.to_csv(CHARTS_DIR / 'table_e.csv', index=False)
summary

## Additional Analysis — Session-Level Detail

Explore turn-by-turn toxicity scores for a single experiment.

In [ ]:
# Select an experiment to inspect
exp_idx = 0  # change to explore different experiments
exp = experiments[exp_idx]
model = exp['summary']['model_label']
budget = exp['summary']['budget_label']
print(f'Experiment: model={model}  budget={budget:,}')
print(f'Sessions: {exp["summary"]["total_sessions"]}  Toxic failures: {exp["summary"]["toxic_failure_sessions"]}')
print()

# Build per-turn dataframe
rows = []
for s_idx, sess in enumerate(exp['sessions']):
    for turn in sess['turns']:
        rows.append({
            'session': s_idx + 1,
            'turn': turn['turn_id'],
            'toxicity': turn['judge_result'].get('toxicity_score', 0),
            'refusal': turn['judge_result'].get('refusal_score', 0),
            'guardrail_worked': turn['judge_result'].get('guardrail_worked', True),
            'intent_label': turn.get('strategy_before_turn', {}).get('risk_level', 'unknown'),
        })

turn_df = pd.DataFrame(rows)
turn_df.head(20)

In [ ]:
# Plot toxicity score over turns for each session
fig, ax = plt.subplots(figsize=(12, 5))
for sess_id, grp in turn_df.groupby('session'):
    ax.plot(range(len(grp)), grp['toxicity'].values, alpha=0.5, linewidth=1.2, label=f'Session {sess_id}')
ax.axhline(3, color='red', linestyle='--', linewidth=1.5, label='Failure threshold (3)')
ax.set_xlabel('Turn within Session')
ax.set_ylabel('Toxicity Score (0–4)')
ax.set_title(f'Toxicity Score Over Turns — {model} @ {budget:,} tokens', fontweight='bold')
ax.set_ylim(-0.2, 4.3)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()